# 8.3 Cube 性能（下）：给定 shape，如何确定最优 K 切分

8.2 讲了「为什么切、切的代价」，本节把流程固化成**拿到 shape 就能执行的选优步骤**。

## 1. 四步流程

```
Step 1  列容量约束        → 求出 kBase 的上限
Step 2  对齐到分形与硬件粒度 → 16（fp16 分形）/ 32B 对齐等
Step 3  建模每个候选 kBase 的耗时 → 搬运 + fixup + 复用损失
Step 4  取耗时最小的 kBase  → 尾块单独兜底
```

### Step 1：容量方程

```
singleCoreM × kBase × dtypeSize ≤ L0A_SIZE
kBase × singleCoreN × dtypeSize ≤ L0B_SIZE
singleCoreM × singleCoreN × dtypeSize ≤ L0C_SIZE   ← 与 K 无关
```

取前两条的较小者，得到 `kBase_max`。

### Step 2：对齐

硬件按分形（fp16 为 16）取数，kBase 必须向下取整到 16 的倍数；同时考虑 MTE1 单笔搬运的粒度（如 32B），保证整块率。

### Step 3：建模

对每个候选 `kBase`，估算三项耗时：

| 项 | 公式 | 随 kBase 的变化 |
|--|--|--|
| 搬运 | `2 × (K/kBase) × (搬运延迟 + 启动开销)` | kBase 越小，笔数越多 |
| fixup | `(K/kBase - 1) × fixup延迟` | 段数越多越贵 |
| 复用损失 | 段内 A 的 N 向分形复用率折算 | kBase 越小损失越大 |

### Step 4：尾块兜底

`K % kBase != 0` 的最后一段走单独的尾块分支（小矩阵乘 + 掩码），不参与建模选优，但**必须保证它不成为流水里的长尾**。

## 2. 一个具体的例子贯穿全流程

以 mm2 常见形态 `M=128, N=128, K=4096`（fp16）为例，先手算约束：

- `128 × kBase × 2 ≤ 64KB → kBase ≤ 256`
- `kBase × 128 × 2 ≤ 64KB → kBase ≤ 256`

所以 kBase 上限是 256，候选集合是 {256, 128, 64, 32, 16}。

## 动手实验：枚举候选并选优


In [ ]:
# 动手：给定 shape 枚举 K 切分方案并建模选优
import numpy as np

M, N, K = 128, 128, 4096
dtype_size = 2
FRACTAL = 16
L0A_SIZE = L0B_SIZE = 64 * 1024
L0C_SIZE = 256 * 1024

# ---- Step 1: 容量约束求上限 ----
k_cap_a = L0A_SIZE // (M * dtype_size)
k_cap_b = L0B_SIZE // (N * dtype_size)
k_max = min(k_cap_a, k_cap_b, K)
print(f"容量约束: kBase <= min({k_cap_a}, {k_cap_b}) = {k_max}")

# ---- Step 2: 对齐到分形 ----
k_max_aligned = (k_max // FRACTAL) * FRACTAL
candidates = [k for k in range(k_max_aligned, 0, -FRACTAL) if k >= FRACTAL]
# 只保留代表性档位，避免输出过长
candidates = [k for k in candidates if k in (256, 128, 64, 32, 16)]

# ---- Step 3: 建模（单位：抽象周期） ----
MTE1_LAT = 100      # 单笔搬运延迟（含启动）
FIXUP_LAT = 60       # 每段部分和的额外开销
CUBE_THR = 0.02      # 每个乘加的 Cube 吞吐周期（示意）

print(f"{'kBase':>6} {'段数':>4} {'搬运周期':>8} {'fixup周期':>9} {'计算周期':>8} {'总计':>8}")
results = {}
for kb in candidates:
    segs = -(-K // kb)
    moves = 2 * segs
    t_move = moves * MTE1_LAT
    t_fix = (segs - 1) * FIXUP_LAT
    t_calc = M * N * K * CUBE_THR          # 计算总量与切分无关
    total = t_move + t_fix + t_calc
    results[kb] = total
    print(f"{kb:>6} {segs:>4} {t_move:>8} {t_fix:>9} {t_calc:>8.0f} {total:>8.0f}")

best = min(results, key=results.get)
print(f"\n>>> 最优 kBase = {best}（切 {(-(-K//best))} 段）")
print(">>> 结论：在容量上限内，kBase 越大越好——搬运和 fixup 都更省")


### 观察结论

在「计算量占大头」的 shape 里，最优解几乎总是**容量上限本身**。这验证了 8.2 的定性判断：**K 切分的优化不是找巧解，而是在容量约束下取最大段**。

## 3. 不同 shape 下的形态切换

再换几组典型 shape 观察最优 kBase 的变化，注意 L0C 约束在什么时候开始「接管」：

In [ ]:
# 动手：多组 shape 的最优 kBase 一览
import numpy as np

dtype_size = 2
FRACTAL = 16
L0A_SIZE = L0B_SIZE = 64 * 1024
L0C_SIZE = 256 * 1024

shapes = [
    (128, 128, 4096, 'mm2: S1块×D, K=S2(训练)'),
    (128, 128, 512,  'mm2: K=S2(短序列)'),
    (64,  128, 128,  'mm1: S1块×S2块, K=D=128'),
    (256, 128, 8192, '大 S1 块 + 长 KV'),
]
for (m, n, k, tag) in shapes:
    k_cap = min(L0A_SIZE // (m * dtype_size), L0B_SIZE // (n * dtype_size))
    k_cap = min(k_cap, k)
    k_cap = (k_cap // FRACTAL) * FRACTAL
    l0c_need = m * n * dtype_size
    l0c_ok = l0c_need <= L0C_SIZE
    print(f"{tag:>28}: M={m:>3} N={n:>3} K={k:>4} -> 最优kBase={k_cap:>4}, "
          f"L0C占用={l0c_need//1024}KB ({'OK' if l0c_ok else '超出!'})")


### 观察结论

1. **mm1（K=D=128）**：容量约束远未触顶，kBase 直接取 128 全载——它根本不需要切；
2. **mm2 长 KV**：约束由 L0A/L0B 决定，最优 kBase = 容量上限；
3. **M 变大时**（如 256）：L0A 约束收紧，同一个 K 需要切成更多段；
4. **L0C 超出时**：只能先切 M/N（这不引入 fixup），而不是硬切 K。

## 4. 与 Tiling 的衔接

工程上这套计算发生在 **Host 侧 Tiling 函数**里（回顾[第3章](../03_fa_code_architecture/README.md)的 Tiling 设计原则）：

- 容量方程、分形对齐、最优 kBase 的求解全部是标量计算 → **Scalar 上提到 Host**，kernel 侧只消费 `TilingData` 里的结果；
- kernel 侧拿到 `kBase` 后，外层 `for (kIdx = 0; kIdx < K; kIdx += kBase)` 循环 + 尾块分支即可。

这也呼应了本章 8.6 的主题：**标量计算能上提就上提**。

## 5. 尾块的三种处理姿势

| 姿势 | 做法 | 适用 |
|--|--|--|
| 掩码 | 尾段照常算，读入侧用 0 填充 | 通用，Vector 侧配合 |
| 降档 | 尾段单独用小 kBase 的模板 | 尾块占比大时 |
| 合并 | 调整切分让尾块并入前一段（牺牲对齐） | 极端不齐时权衡 |

## 本节要点

- 选优四步：容量约束 → 分形对齐 → 建模 → 尾块兜底；
- 建模三笔账：搬运笔数、fixup 段数、复用率；计算量与切分无关；
- **最优 kBase ≈ 容量上限本身**；L0C 满时应先切 M/N 而非硬切 K；
- 整套计算在 Host Tiling 侧完成，kernel 只消费结果。

## 小测验

1. 写出 kBase 的 L0A/L0B 容量约束方程，并说明为什么 L0C 不参与该方程。
2. `M=128, N=128, K=2048, fp16, L0A=L0B=64KB` 时 kBase 上限是多少？
3. 为什么说「K 切分的优化不是找巧解，而是取容量上限」？
4. K=4100、kBase=256 时怎么处理尾段？列出至少两种方案。

>>> 参考答案见 [answer/08.03_answer.txt](answer/08.03_answer.txt)
